In [ ]:
import jax.numpy as jnp
from scipy.stats import norm, multivariate_normal, binom
from IPython.display import Markdown, display
from jax import random

In [ ]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

## Classic kernels

The **squared exponential** covariance function is given by

\begin{align*} 
    k_{\text{SE}}(\mathbf{x}_n, \mathbf{x}_m) = \kappa^2 \exp\left(-\frac{\|\mathbf{x}_n - \mathbf{x}_m\|^2_ 2}{2\ell^2}\right), \tag{3}
\end{align*}

where $\kappa > 0$ and $\ell > 0$ are hyperparameters of the kernel.

**Matérn kernels**

$$\begin{align*}
k_{\text{Matern12}}(\mathbf{x}_n, \mathbf{x}_m) &= \kappa^2 \exp\left(-\frac{||\mathbf{x}_n- \mathbf{x}_m||}{\ell}\right)\\
k_{\text{Matern32}}(\mathbf{x}_n, \mathbf{x}_m) &= \kappa^2 \left(1 + \frac{\sqrt{3}||\mathbf{x}_n - \mathbf{x}_m||}{\ell}\right)\exp\left(-\frac{\sqrt{3}||\mathbf{x}_n- \mathbf{x}_m||}{\ell}\right).
\end{align*}
$$

# Gaussian Process Regression

## Dictionary

$\bf{X} = \text{Training points}$
$x^{*} = \text{Test points}$

Covariance matrix between training points:
$$\bf{K} = k(X,X)$$

Covariance matrix between test points:
$$\bf{K_{*}} = k(x_{*},x_{*})$$

Covariance matrix between test points and training points:
$$\bf{k} = k(\bf{x_{*}},\bf{X})$$

Covariance between test points with added noise from likelihood:
$$c = \bf{k}(\bf{x_{*}}, \bf{x_{*}}) + \beta^{-1}$$

In [ ]:
#Define training data
xtrain = jnp.array([-2.17, 1.99 ,0.57,-3.01,-1.16, 3.30,-4.85,-0.86])[:, None]

#Define training targets
ytrain = jnp.array([0.88, 0.46,-0.06, 0.98, 0.45, 0.88,-0.66, 0.05])[:, None]

#Define test data
x_star = jnp.array([1])[:, None]

#Define design matrix 
def design_matrix(x):
    return jnp.column_stack((jnp.ones(len(x)), x))

kappa = 0.7
lengthscale = jnp.sqrt(2) / 2
sigma = 1/5

kernel = lambda x, xs: kappa**2*jnp.exp(-0.5*(x-xs)**2/lengthscale**2)

def kernel_func(x,xs):
    return kernel(x.T,xs)

k = kernel_func(x_star, xtrain)
K = kernel_func(xtrain, xtrain)
Kstar = kernel_func(x_star, x_star)
c = Kstar + jnp.eye(len(x_star)) * sigma**2


## Prior predictive distribution - $p(y^{*} | x^{*})$

### Equation

$$p(y^{*} | x^{*}) = \mathcal{N}(y^{*} | 0, k_{1}(x^{*}, x^{*}) + \sigma^{2})$$

### Code

In [ ]:
prior_mu = 0
prior_var = kernel_func(x_star, x_star) + sigma**2

display(Markdown(rf"The prior predictive distribution is $p(y_* \mid x_*) = \mathcal{{N}}(y_* \mid \mu_*, \sigma_*^2)$ where $\mu_* = {prior_mu}$ and $\sigma_*^2 = {to_latex_matrix(prior_var)}$"))

The prior predictive distribution is $p(y_* \mid x_*) = \mathcal{N}(y_* \mid \mu_*, \sigma_*^2)$ where $\mu_* = 0$ and $\sigma_*^2 = \begin{bmatrix}0.53\end{bmatrix}$

## Posterior predictive distribution - $p(f^{*} | \bf{y}, x^{*})$

### Equation

$$p(f^{*} | \bf{y}) = \mathcal{N}(f^{*} | \mu_{f^{*} | \bf{y}}, \sigma^{2}_{f^{*} | \bf{y}})$$
where

$$\mu_{f^* \mid y} = \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{y}$$
and
$$\sigma^2_{f^* \mid y} = \bf{K_{*}} - \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{k}^T$$


### Code

In [ ]:
C = K + jnp.eye(len(xtrain)) * sigma**2
post_mu_f = jnp.dot(k.T,jnp.linalg.solve(C, ytrain))
post_var_f = Kstar - jnp.dot(k.T,jnp.linalg.solve(C, k))

display(Markdown(rf"The posterior predictive distribution is $p(f_* \mid x_*, \bf{{y}}) = \mathcal{{N}}(f_* \mid \mu_{{f^{{*}} | y}}, \sigma_{{f^{{*}} | y}}^2)$ where $\mu_* = {to_latex_matrix(post_mu_f)}$ and $\sigma_*^2 = {to_latex_matrix(post_var_f)}$"))

The predictive posterior distribution is $p(y_* \mid x_*, \bf{y}) = \mathcal{N}(y_* \mid \mu_{y^{*} | y}, \sigma_{y^{*} | y}^2)$ where $\mu_* = \begin{bmatrix}0.07\end{bmatrix}$ and $\sigma_*^2 = \begin{bmatrix}0.14\end{bmatrix}$

## Posterior predictive distribution - $p(y^{*} | \bf{y}, x^{*})$

### Equation

$$p(y^{*} | \bf{y}) = \mathcal{N}(y^{*} | \mu_{y^{*} | \bf{y}}, \sigma^{2}_{y^{*} | \bf{y}})$$
where

$$\mu_{y^* \mid y} = \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{y}$$
and
$$\sigma^2_{y^* \mid y} = c - \mathbf{k} \left( \mathbf{K} + \beta^{-1} \mathbf{I} \right)^{-1} \mathbf{k}^T$$


### Code:

In [190]:
C = K + jnp.eye(len(xtrain)) * sigma**2
post_mu_y = jnp.dot(k.T,jnp.linalg.solve(C, ytrain))
post_var_y = c - jnp.dot(k.T,jnp.linalg.solve(C, k))

display(Markdown(rf"The predictive posterior distribution is $p(y_* \mid x_*, \bf{{y}}) = \mathcal{{N}}(y_* \mid \mu_{{y^{{*}} | y}}, \sigma_{{y^{{*}} | y}}^2)$ where $\mu_* = {to_latex_matrix(post_mu_y)}$ and $\sigma_*^2 = {to_latex_matrix(post_var_y)}$"))

The predictive posterior distribution is $p(y_* \mid x_*, \bf{y}) = \mathcal{N}(y_* \mid \mu_{y^{*} | y}, \sigma_{y^{*} | y}^2)$ where $\mu_* = \begin{bmatrix}0.07\end{bmatrix}$ and $\sigma_*^2 = \begin{bmatrix}0.18\end{bmatrix}$

## Marginal likelihood - $p(\bf{y} | \theta)$

### Equation

$$p(\bf{y}| \theta) = \mathcal{N}(\bf{y} | \bf{0}, \beta^{-1} \bf{I} + \bf{K})$$

### Code

In [ ]:
ml_var = sigma**2 * jnp.eye(len(xtrain)) + K
ml_mu = jnp.zeros(len(xtrain))


display(Markdown(
    rf"The marginal likelihood is $p(\mathbf{{y}} \mid \theta) = "
    rf"\mathcal{{N}}(\mathbf{{y}} \mid \mathbf{{0}}, \beta^{{-1}}\mathbf{{I}} + \mathbf{{K}})$"
    rf" where $\beta^{{-1}}\mathbf{{I}} + \mathbf{{K}} = {to_latex_matrix(ml_var)}$"
))

The marginal likelihood is $p(\mathbf{y} \mid \theta) = \mathcal{N}(\mathbf{y} \mid \mathbf{0}, \beta^{-1}\mathbf{I} + \mathbf{K})$ where $\beta^{-1}\mathbf{I} + \mathbf{K} = \begin{bmatrix}0.53 & 0.00 & 0.00 & 0.24 & 0.18 & 0.00 & 0.00 & 0.09 \\ 0.00 & 0.53 & 0.07 & 0.00 & 0.00 & 0.09 & 0.00 & 0.00 \\ 0.00 & 0.07 & 0.53 & 0.00 & 0.02 & 0.00 & 0.00 & 0.06 \\ 0.24 & 0.00 & 0.00 & 0.53 & 0.02 & 0.00 & 0.02 & 0.00 \\ 0.18 & 0.00 & 0.02 & 0.02 & 0.53 & 0.00 & 0.00 & 0.45 \\ 0.00 & 0.09 & 0.00 & 0.00 & 0.00 & 0.53 & 0.00 & 0.00 \\ 0.00 & 0.00 & 0.00 & 0.02 & 0.00 & 0.00 & 0.53 & 0.00 \\ 0.09 & 0.00 & 0.06 & 0.00 & 0.45 & 0.00 & 0.00 & 0.53\end{bmatrix}$

## Log marginal likelihood - $\log{p(\bf{y} | \theta)}$

### Equation

$$\ln p(\mathbf{y}|\boldsymbol{\theta}) = -\frac{N}{2}\ln(2\pi) - \frac{1}{2}\ln|\beta^{-1}\mathbf{I} + \mathbf{K}| - \frac{1}{2}\mathbf{y}^T\left(\beta^{-1}\mathbf{I} + \mathbf{K}\right)^{-1}\mathbf{y}$$

### Code

In [ ]:
const_term = -len(xtrain) /2 * jnp.log(2*jnp.pi)
log_det_term = -1/2 * jnp.log(jnp.linalg.det(C)) 
quad_term = -1/2 * jnp.dot(ytrain.T, jnp.linalg.solve(C, ytrain))

lm_likelihood = const_term + log_det_term + quad_term
display(Markdown(
    rf"The log marginal likelihood is "
    rf"$\ln p(\mathbf{{y}} \mid \boldsymbol{{\theta}}) = "
    rf"-\frac{{N}}{{2}}\ln(2\pi) - \frac{{1}}{{2}}\ln|\beta^{{-1}}\mathbf{{I}} + \mathbf{{K}}| - \frac{{1}}{{2}}\mathbf{{y}}^T(\beta^{{-1}}\mathbf{{I}} + \mathbf{{K}})^{{-1}}\mathbf{{y}}$"
    rf" $= {const_term:.4f} + ({log_det_term:.4f}) + ({to_latex_matrix(quad_term)}) = {to_latex_matrix(lm_likelihood)}$"
))

The log marginal likelihood is $\ln p(\mathbf{y} \mid \boldsymbol{\theta}) = -\frac{N}{2}\ln(2\pi) - \frac{1}{2}\ln|\beta^{-1}\mathbf{I} + \mathbf{K}| - \frac{1}{2}\mathbf{y}^T(\beta^{-1}\mathbf{I} + \mathbf{K})^{-1}\mathbf{y}$ $= -7.3515 + (3.4183) + (\begin{bmatrix}-2.72\end{bmatrix}) = \begin{bmatrix}-6.65\end{bmatrix}$

## Implementation of new kernel

### Write kernel here:

In [ ]:
#Parameters
c1 = 1
c2 = 1
lengthscale = 1/jnp.sqrt(2)

#Kernel
kernel2 = lambda x, xs: c1 * (1 + jnp.abs(x - xs) / (2*lengthscale**2))**(-1) + c2*x*xs

def kernel_func2(x,xs):
    return kernel2(x.T,xs)

### Compute new quantities here

In [193]:
#New xstar
xstar = jnp.array([-1])[:,None]

kernel_func2(xtrain,xstar)

Array([[ 2.6308296 , -1.7393734 , -0.18089497,  3.3422258 ,  2.022069  ,
        -3.1113207 ,  5.0561852 ,  1.737193  ]], dtype=float32)

## Stationary and isotropic

If a kernel is a **stationary** kernel, it means that the covariance function only depends on the difference between two points, i.e. $\tau = \mathbf{x}_n - \mathbf{x}_m$ for two points in the input space $\mathbf{x}_n, \mathbf{x}_m \in \mathbb{R}^D$

For a kernel to be isotropic the covariance function only depends on the distance between any two inputs, i.e. $||\tau||_2 = ||\mathbf{x}_n - \mathbf{x}_m||_2$.
